# Step 1 — Crop cells out of the raw 4D movies

Lattice light-sheet acquisitions contain several cells per field of view. This
notebook cuts each hand-drawn ROI out of every timepoint and writes it as an
independent single-cell movie, which is what step 2 expects as input.

**Input** — one folder per sample under `RAW_DATA_DIR`, containing

* one deskewed/deconvolved TIFF per timepoint per channel, and
* an ImageJ ROI set (`*.zip`, one rectangular ROI per cell) saved in that folder.

**Output** — under `SAVE_ROOT/<experiment>/<condition>/roi_id_<n>/<channel>/frame_<t>.tif`

ROI ids are numbered continuously within a condition, so cells coming from
different samples of the same condition never collide.

In [ ]:
import glob
import os
import time

import roifile
from tifffile import imread, imwrite
from tqdm.notebook import tqdm, trange

## Configuration

Everything that changes between experiments lives in this cell.

In [ ]:
# Where the microscope writes its processed output, and where crops should go.
RAW_DATA_DIR = "/path/to/MOSAIC_Data/Processed_Data"
SAVE_ROOT = "/path/to/MOSAIC_Data/Analyzed_Data"

EXPERIMENT_NAME = "Pham THP1"

# Glob (relative to RAW_DATA_DIR) selecting the sample folders of this experiment.
SAMPLE_GLOB = "*1121*Pham*/Sample 1/*"

# Filename pattern per channel. Green is the deconvolved mitochondrial marker;
# red is the un-deconvolved mitophagy (mitolysosome) marker, kept raw so that
# step 2 fits blobs to the measured PSF rather than to deconvolution artefacts.
CHANNELS = {
    "green": "*488nm*processed*tif",
    "red": "*560nm*no_decon*tif",
}

# ImageJ ROIs are 2D, so the axial extent of the crop is set once by hand after
# inspecting the stack: keep the slices that actually contain cells.
LOW_Z, HIGH_Z = 50, 220

## Select the samples to process

In [ ]:
all_sample_paths = sorted(glob.glob(os.path.join(RAW_DATA_DIR, SAMPLE_GLOB)))

for i, path in enumerate(all_sample_paths):
    print(i, os.path.relpath(path, RAW_DATA_DIR))

In [ ]:
# Process every sample found above...
selected_paths = all_sample_paths

# ...or pick a subset by index while testing:
# selected_paths = [all_sample_paths[i] for i in [0, 1, 2]]

print(len(selected_paths), "sample(s) selected")

## Assign a condition to each sample

`CONDITION_NAMES` must be in the same order as `selected_paths`. The six
repeats per condition below are the acquisition order of this experiment.

In [ ]:
CONDITION_NAMES = (
    ["PGE2-activator"] * 6
    + ["Control"] * 6
    + ["PGE2-activator-EP4-inhibitor"] * 6
    + ["PGE2-activator-PKA-inhibitor"] * 6
)

assert len(CONDITION_NAMES) == len(selected_paths), (
    f"{len(CONDITION_NAMES)} condition labels for {len(selected_paths)} samples"
)

for path, condition in zip(selected_paths, CONDITION_NAMES):
    print(os.path.relpath(path, RAW_DATA_DIR), "->", condition)

## Crop

For every sample, every channel and every timepoint: read the full stack once,
then write one cropped TIFF per ROI. Reading dominates the runtime, hence the
reported read speed — a full stack is several hundred MB.

Run this on a single sample first (`selected_paths = all_sample_paths[:1]`) and
check the crops in Fiji or napari before launching the whole experiment.

In [ ]:
# Next free ROI id per condition, so ids stay unique across samples.
next_roi_id = {condition: 0 for condition in CONDITION_NAMES}

for sample_path, condition in zip(tqdm(selected_paths, desc="samples"), CONDITION_NAMES):

    roi_zips = glob.glob(os.path.join(sample_path, "*zip"))
    if not roi_zips:
        print("No ROI set found, skipping", sample_path)
        continue

    rois = roifile.ImagejRoi.fromfile(roi_zips[0])
    if len(rois) < 1:
        print("Empty ROI set, skipping", sample_path)
        continue

    condition_dir = os.path.join(SAVE_ROOT, EXPERIMENT_NAME, condition)

    for channel, pattern in CHANNELS.items():
        tif_paths = sorted(glob.glob(os.path.join(sample_path, pattern)))

        for frame in trange(len(tif_paths), desc=f"{condition} / {channel}", leave=False):
            start = time.time()
            full_stack = imread(tif_paths[frame])
            megabytes = full_stack.nbytes / 1e6
            print(
                f"{os.path.basename(tif_paths[frame])} "
                f"({megabytes:.0f} MB at {megabytes / (time.time() - start):.1f} MB/s)"
            )

            for offset, roi in enumerate(rois):
                roi_id = next_roi_id[condition] + offset
                out_dir = os.path.join(condition_dir, f"roi_id_{roi_id}", channel)
                os.makedirs(out_dir, exist_ok=True)

                cropped = full_stack[LOW_Z:HIGH_Z, roi.top:roi.bottom, roi.left:roi.right]
                imwrite(os.path.join(out_dir, f"frame_{frame}.tif"), cropped)

    next_roi_id[condition] += len(rois)

print("Done:", {c: n for c, n in next_roi_id.items()}, "cells per condition")